# Lab Assignment 9

**Student roll number:** 1024170115

## Part A. Medicine bottle label reader

In [ ]:
!pip -q install easyocr opencv-python-headless

In [ ]:
from pathlib import Path
import cv2, easyocr, matplotlib.pyplot as plt
from google.colab import files

uploaded = files.upload()
image_path = next(iter(uploaded))
image = cv2.imread(image_path)
if image is None:
    raise ValueError("Could not read the uploaded image.")
plt.figure(figsize=(8,6)); plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()
reader = easyocr.Reader(["en"], gpu=False)
results = reader.readtext(image)
recognized_lines = [text for box, text, confidence in results if confidence >= 0.25]
print("Recognized label text:")
print("\n".join(recognized_lines))

In [ ]:
import re
label_text = "\n".join(recognized_lines)
fields = {
    "expiry_dates": re.findall(r"\b(?:0?[1-9]|1[0-2])[/-](?:20)?\d{2}\b|\b\d{2}[/-](?:0?[1-9]|1[0-2])[/-]\d{2,4}\b", label_text),
    "batch_numbers": re.findall(r"(?i)\b(?:batch|lot)\s*(?:no\.?|number)?\s*[:#-]?\s*([A-Z0-9-]+)", label_text),
    "dosage_mentions": re.findall(r"\b\d+(?:\.\d+)?\s?(?:mg|mcg|g|ml|%)\b", label_text, flags=re.I),
}
print(fields)
print("OCR can misread small or curved print. Confirm extracted fields against the original packaging.")

## Part B. Emotion-aware assistant using facial expression detection

In [ ]:
!pip -q install fer opencv-python-headless

In [ ]:
from fer import FER
from google.colab import files
import cv2, matplotlib.pyplot as plt
uploaded = files.upload()
face_image_path = next(iter(uploaded))
face_image = cv2.imread(face_image_path)
if face_image is None:
    raise ValueError("Could not read the uploaded image.")
plt.figure(figsize=(6,6)); plt.imshow(cv2.cvtColor(face_image, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()
detector = FER(mtcnn=False)
emotions = detector.detect_emotions(face_image)
if not emotions:
    print("No face detected. Try an image with one clearly visible face.")
else:
    scores = emotions[0]["emotions"]
    emotion = max(scores, key=scores.get)
    confidence = scores[emotion]
    print("Most likely expression label:", emotion, "score:", round(confidence, 3))
    print("Assistant response: I can offer a supportive check-in. Would you like to talk about how you are feeling?")

## Summary
The label reader extracts text and a few common fields. The second prototype detects a face and estimates an expression category. Both systems can fail on poor images, and neither should replace a pharmacist, clinician, or the person’s own account of their needs.